|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 1:</h2>|<h1>The Naive Loop<h1>|
|<h2>Section:</h2>|<h1>The roofline<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: prefill and decode are two different machines<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

The last notebook put a matmul on the roofline. Now put a real model on it.

One model does two jobs. Prefill reads a whole prompt at once. Decode makes
one token at a time. You will find them on opposite sides of the ridge. Almost
everything else in this course follows from that.

In [ ]:
### run this cell: the model, and what it weighs

model = AutoModelForCausalLM.from_pretrained(
          'Qwen/Qwen3-0.6B', dtype=torch.bfloat16).cuda().eval()

weight_bytes = sum(p.numel()*p.element_size() for p in model.parameters())
bandwidth    = cudalib.peak_bandwidth(fresh=True)

print(f'weights:   {weight_bytes/1e9:.2f} GB')
print(f'bandwidth: {bandwidth:.0f} GB/s')
print(f'so one read of the weights costs at least {1000*weight_bytes/1e9/bandwidth:.2f} ms')

# Exercise 1: how fast is prefill

Time a forward pass over a prompt of length L. Use no cache. Then report the
cost per token.

In [ ]:
@torch.inference_mode()
def prefill_ms(L):
  x = torch.randint(0, 1000, (1,L), device='cuda')
  return 

print(f"{'prompt':>7} {'ms':>9} {'ms/token':>10}")
for L in [128,256,512,1024,2048]:
  ms = prefill_ms(L)
  print(f'{L:>7} {ms:>9.2f} {ms/L:>10.4f}')

# Exercise 2: how fast is decode

Now time one extra token. The context is already in the cache. A server does
this for every token after the first.

In [ ]:
import copy

@torch.inference_mode()
def decode_ms(L):
  x    = torch.randint(0, 1000, (1,L), device='cuda')
  past = model(x, use_cache=True).past_key_values
  nxt  = torch.randint(0, 1000, (1,1), device='cuda')
  # time ONE more token, with the context already cached.
  # copy.copy(past) keeps each timed call starting from the same state.
  return 

print(f"{'context':>8} {'ms per token':>13}")
for L in [128,512,2048]:
  print(f'{L:>8} {decode_ms(L):>13.2f}')

# Exercise 3: against the floor

A decode step must read every weight. That is a hard floor in milliseconds.
Compute the floor. Compare it with your measurement. Then turn the difference
into an achieved-bandwidth number.

In [ ]:
p_ms = prefill_ms(2048)/2048
d_ms = decode_ms(2048)

# a decode step cannot beat one read of the weights. What is that floor?
floor_ms = 

# and how many bytes per second did you actually get?
achieved = 

print(f'prefill: {p_ms:8.4f} ms/token')
print(f'decode:  {d_ms:8.4f} ms/token   ({d_ms/p_ms:.0f}x more, for the same model)')
print(f'\nthe floor for a decode step is {floor_ms:.2f} ms (one read of the weights)')
print(f'you measured                   {d_ms:.2f} ms')
print(f'achieved bandwidth             {achieved:.0f} GB/s = {100*achieved/bandwidth:.0f}% of peak')

### Before you open the solution

Answer these questions from your own numbers:

1. Decode costs far more per token than prefill. The weights are the same, and
   the arithmetic per weight is the same. So what differs?
2. Your decode step is slower than one read of the weights, so the achieved
   bandwidth falls below peak. Name something other than the memory system
   that a 0.6B model waits for. Which stage removes it?
3. Now assume that you removed it completely. The floor remains. What is the
   only way to make a token faster than the floor?